# 02 - Träna och inferera

Det här notebooket gör fyra saker:
- jämför `k=2,3,4` med silhouette score
- tränar en pipeline per `k`
- plottar kluster i 2D (`recency_days` mot `monetary`)
- plottar nya inference-punkter i samma figur


In [ ]:
from pathlib import Path
import sqlite3
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / 'data').exists() and (root.parent / 'data').exists():
    root = root.parent
if not (root / 'data').exists() and (root.parent.parent / 'data').exists():
    root = root.parent.parent

figures_dir = root / 'outputs' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
models_dir = root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
train_df = pd.read_csv(root / 'data' / 'processed' / 'customer_enriched.csv', parse_dates=['last_purchase_date'])
train_df.head()


## Silhouette score kort förklaring

Silhouette score mäter hur tydligt en punkt ligger i sitt eget kluster jämfört med närmaste andra kluster.

- nära `1.0`: tydlig separation
- nära `0.0`: kluster överlappar
- under `0.0`: många punkter ligger sannolikt i fel kluster

Vi använder detta som stöd vid val av `k`, inte som absolut facit.


In [ ]:
numeric_features = [
    'recency_days',
    'frequency',
    'monetary',
    'avg_order_value',
    'basket_size',
    'avg_items_per_invoice',
    'unique_products',
    'avg_unit_price',
]
categorical_features = ['RegionGroup']
feature_columns = numeric_features + categorical_features
X_train = train_df[feature_columns].copy()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Pipeline kopplar ihop samma transformering med modellen,
# så att träning och inference går via exakt samma steg.
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])


In [ ]:
k_values = [2, 3, 4]
model_selection_rows = []
trained_pipelines = {}
train_cluster_by_k = {}

for k in k_values:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('kmeans', KMeans(n_clusters=k, random_state=42, n_init=20)),
    ])

    train_clusters = pipeline.fit_predict(X_train)
    X_prepared = pipeline.named_steps['preprocessor'].transform(X_train)
    score = silhouette_score(X_prepared, train_clusters)

    trained_pipelines[k] = pipeline
    train_cluster_by_k[k] = train_clusters
    model_selection_rows.append({'k': k, 'silhouette_score': score})

model_selection_df = pd.DataFrame(model_selection_rows).sort_values('k')
model_selection_df


In [ ]:
best_k = int(model_selection_df.loc[model_selection_df['silhouette_score'].idxmax(), 'k'])

sns.set_theme(style='whitegrid', context='talk')
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=model_selection_df, x='k', y='silhouette_score', marker='o', ax=ax, color='#0b5cad')
ax.axvline(best_k, color='#d62728', linestyle='--', linewidth=1)
ax.set_title('Silhouette score by number of clusters')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.set_xticks(k_values)
for _, row in model_selection_df.iterrows():
    ax.annotate(f"{row['silhouette_score']:.3f}", (row['k'], row['silhouette_score']), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(figures_dir / 'silhouette_scores.png', dpi=160, bbox_inches='tight')
plt.show()

print(f'Valt k enligt h?gst silhouette: {best_k}')


In [ ]:
new_customers_df = pd.read_csv(root / 'data' / 'raw' / 'new_customers.csv')
regions_df = pd.read_csv(root / 'data' / 'raw' / 'regions.csv')

# RegionGroup sätts via lookup-join, inte via modellprediktion.
conn = sqlite3.connect(':memory:')
new_customers_df.to_sql('new_customers', conn, index=False, if_exists='replace')
regions_df.to_sql('regions', conn, index=False, if_exists='replace')

inference_df = pd.read_sql_query(
    """
    SELECT
        n.*,
        COALESCE(r.RegionGroup, 'Other') AS RegionGroup
    FROM new_customers n
    LEFT JOIN regions r
        ON n.Country = r.Country
    """,
    conn,
)

# Samma pipelineobjekt används här, så nya rader får samma transformering.
for k in k_values:
    inference_df[f'PredictedCluster_k{k}'] = trained_pipelines[k].predict(inference_df[feature_columns])

train_output_df = train_df.copy()
for k in k_values:
    train_output_df[f'Cluster_k{k}'] = train_cluster_by_k[k]

best_pipeline = trained_pipelines[best_k]
joblib.dump(best_pipeline, models_dir / 'clustering_pipeline.joblib')
model_selection_df.to_csv(root / 'outputs' / 'model_selection.csv', index=False)
train_output_df.to_csv(root / 'outputs' / 'clustered_customers.csv', index=False)
inference_df.to_csv(root / 'outputs' / 'inference_results.csv', index=False)

inference_df[['CustomerID', 'Country', 'PredictedCluster_k2', 'PredictedCluster_k3', 'PredictedCluster_k4']].head()


## Kort analys av kluster (k = 3)

KMeans grupperar kunder efter likhet i *hela* feature-rummet efter preprocessing
(imputering, log-skalning, standardisering och one-hot av region).

Det betyder att klustren är separerade statistiskt i den transformerade rymden,
men inte nödvändigtvis helt separerade i en enskild 2D-plot.


In [ ]:
k3_profile_df = (
    train_output_df
    .groupby('Cluster_k3')
    .agg(
        customers=('CustomerID', 'count'),
        recency_mean=('recency_days', 'mean'),
        frequency_mean=('frequency', 'mean'),
        monetary_mean=('monetary', 'mean'),
        avg_order_value_mean=('avg_order_value', 'mean'),
    )
    .round(2)
)
k3_profile_df['share_of_customers'] = (k3_profile_df['customers'] / len(train_output_df)).round(3)
k3_profile_df = k3_profile_df[['customers', 'share_of_customers', 'recency_mean', 'frequency_mean', 'monetary_mean', 'avg_order_value_mean']]
k3_profile_df

def level_text(value, low, high):
    if value <= low:
        return 'låg'
    if value <= high:
        return 'medel'
    return 'hög'

rec_low, rec_high = k3_profile_df['recency_mean'].quantile([1/3, 2/3])
freq_low, freq_high = k3_profile_df['frequency_mean'].quantile([1/3, 2/3])
mon_low, mon_high = k3_profile_df['monetary_mean'].quantile([1/3, 2/3])

print('Kort tolkning av k=3-kluster:')
print('- Recency tolkas som dagar sedan senaste köp: låg recency = mer nyliga kunder.')
for cluster_id, row in k3_profile_df.iterrows():
    rec_level = level_text(row['recency_mean'], rec_low, rec_high)
    freq_level = level_text(row['frequency_mean'], freq_low, freq_high)
    mon_level = level_text(row['monetary_mean'], mon_low, mon_high)
    print(
        f"- Kluster {cluster_id}: recency {rec_level}, frekvens {freq_level}, monetärt värde {mon_level}."
    )


## Utökad klusterprofil (k = 3)

Här tittar vi på fler features och visualiserar dem som ett enkelt stapeldiagram.
Staplarna visas som *index mot totalen* (1.0 = samma nivå som hela datasetet).


In [ ]:
k3_extended_profile_df = (
    train_output_df
    .groupby('Cluster_k3')
    .agg(
        frequency=('frequency', 'mean'),
        monetary=('monetary', 'mean'),
        avg_order_value=('avg_order_value', 'mean'),
        basket_size=('basket_size', 'mean'),
        unique_products=('unique_products', 'mean'),
        avg_items_per_invoice=('avg_items_per_invoice', 'mean'),
    )
    .round(2)
)
k3_extended_profile_df

plot_features = [
    'frequency',
    'monetary',
    'avg_order_value',
    'basket_size',
    'unique_products',
    'avg_items_per_invoice',
]
feature_label_map = {
    'frequency': 'Frequency',
    'monetary': 'Monetary',
    'avg_order_value': 'Avg order value',
    'basket_size': 'Basket size',
    'unique_products': 'Unique products',
    'avg_items_per_invoice': 'Avg items/invoice',
}

k3_relative_df = k3_extended_profile_df[plot_features].div(train_output_df[plot_features].mean(), axis=1)
k3_relative_long_df = (
    k3_relative_df
    .reset_index()
    .melt(id_vars='Cluster_k3', var_name='feature', value_name='relative_index')
)
k3_relative_long_df['feature_label'] = k3_relative_long_df['feature'].map(feature_label_map)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=k3_relative_long_df,
    x='feature_label',
    y='relative_index',
    hue='Cluster_k3',
    palette='tab10',
)
plt.axhline(1.0, color='black', linestyle='--', linewidth=1)
plt.title('k=3 cluster profile across multiple features (index vs total)')
plt.xlabel('Feature')
plt.ylabel('Relative index (1.0 = overall mean)')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Cluster', loc='upper right')
plt.tight_layout()
plt.savefig(figures_dir / 'k3_cluster_feature_bars.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
plot_x = 'recency_days'
plot_y = 'monetary'

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, k in zip(axes, k_values):
    cluster_order = list(range(k))
    cluster_colors = dict(zip(cluster_order, sns.color_palette('tab10', k)))

    plot_train_df = train_output_df[[plot_x, plot_y, f'Cluster_k{k}']].rename(columns={f'Cluster_k{k}': 'Cluster'})
    plot_inference_df = inference_df[[plot_x, plot_y, f'PredictedCluster_k{k}']].rename(columns={f'PredictedCluster_k{k}': 'Cluster'})

    # hue_order + samma palette förhindrar att inference-punkter får fel färg.
    sns.scatterplot(
        data=plot_train_df,
        x=plot_x,
        y=plot_y,
        hue='Cluster',
        hue_order=cluster_order,
        palette=cluster_colors,
        alpha=0.5,
        s=30,
        legend=False,
        ax=ax,
    )

    sns.scatterplot(
        data=plot_inference_df,
        x=plot_x,
        y=plot_y,
        hue='Cluster',
        hue_order=cluster_order,
        palette=cluster_colors,
        marker='X',
        s=170,
        edgecolor='black',
        linewidth=0.8,
        legend=False,
        ax=ax,
    )

    ax.set_yscale('log')
    ax.set_title(f'k = {k}')
    ax.set_xlabel('Recency (days)')

axes[0].set_ylabel('Monetary (log scale)')
fig.suptitle('Clusters with inference points (X) for k = 2, 3, 4', fontsize=16)
plt.tight_layout()
plt.savefig(figures_dir / 'cluster_scatter_k234_with_inference.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
best_preprocessor = trained_pipelines[best_k].named_steps['preprocessor']
X_train_prepared = best_preprocessor.transform(X_train)
X_inference_prepared = best_preprocessor.transform(inference_df[feature_columns])

pca_model = PCA(n_components=2, random_state=42)
train_pca_values = pca_model.fit_transform(X_train_prepared)
inference_pca_values = pca_model.transform(X_inference_prepared)

train_pca_df = pd.DataFrame(train_pca_values, columns=['PC1', 'PC2'])
inference_pca_df = pd.DataFrame(inference_pca_values, columns=['PC1', 'PC2'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, k in zip(axes, k_values):
    cluster_order = list(range(k))
    cluster_colors = dict(zip(cluster_order, sns.color_palette('tab10', k)))

    plot_train_pca_df = train_pca_df.copy()
    plot_train_pca_df['Cluster'] = train_output_df[f'Cluster_k{k}'].values

    plot_inference_pca_df = inference_pca_df.copy()
    plot_inference_pca_df['Cluster'] = inference_df[f'PredictedCluster_k{k}'].values

    sns.scatterplot(
        data=plot_train_pca_df,
        x='PC1',
        y='PC2',
        hue='Cluster',
        hue_order=cluster_order,
        palette=cluster_colors,
        alpha=0.5,
        s=30,
        legend=False,
        ax=ax,
    )

    sns.scatterplot(
        data=plot_inference_pca_df,
        x='PC1',
        y='PC2',
        hue='Cluster',
        hue_order=cluster_order,
        palette=cluster_colors,
        marker='X',
        s=170,
        edgecolor='black',
        linewidth=0.8,
        legend=False,
        ax=ax,
    )

    ax.set_title(f'k = {k}')
    ax.set_xlabel('PCA component 1')

axes[0].set_ylabel('PCA component 2')
fig.suptitle('Clusters with inference points (X) in PCA space for k = 2, 3, 4', fontsize=16)
plt.tight_layout()
plt.savefig(figures_dir / 'cluster_scatter_pca_k234_with_inference.png', dpi=160, bbox_inches='tight')
plt.show()


## Kommentar och slutsats

- Silhouette används här för att jämföra `k=2,3,4` på ett enkelt och reproducerbart sätt.
- I den här datan blir separationen tydligast för lägre `k`, men fler kluster kan fortfarande vara användbara om man vill ha mer detaljerad segmentering.
- 2D-plotten med `recency_days` mot `monetary` är lätt att tolka affärsmässigt.
- PCA-plotten ger en kompletterande vy av samma klusterstruktur i ett reducerat feature-rum.
- Inference-punkterna markeras med `X` i båda vyerna för att visa var nya kunder hamnar i respektive modell.
